# Projeto MedIR-RAG — Milestone 2 (Recuperação Densa com FAISS)
## Comparação Lexical (BM25 da M1) vs Recuperação Densa Zero-Shot (Bi-Encoder + FAISS)

Versão focada e limpa que integra exclusivamente o que foi pedido:
1. **Carregamento e Split Fixo (Train/Test):** Reutilização da coleção baseada em snippets e fixação do conjunto de teste (seed 13) para garantir comparabilidade com a Milestone 1.
2. **Baseline BM25:** Avaliação com a tua baseline da Milestone 1.
3. **Recuperação Densa com FAISS:** Exatamente a classe `FAISSDenseRetriever` e o Bi-Encoder (`BAAI/bge-small-en-v1.5`) do exemplo do professor.
4. **Tabela Comparativa Simples:** Comparação direta de métricas (`NDCG@5`, `NDCG@10`, `MRR@10`, `MAP`, `Recall@100`) calculada em Python/NumPy (compatível com Windows, sem erros de compilação).

### 1. Instalação e Imports

In [13]:
%pip install -q faiss-cpu sentence-transformers rank_bm25 pandas numpy torch tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import faiss
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

# Fixação de seeds para reprodutibilidade
SEED = 13
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo em utilização: {device}")

Dispositivo em utilização: cpu


### 2. Carregamento dos Dados e Preparação das Estruturas
Carregamos o ficheiro `BioASQ-trainingDataset2b.json` da Milestone 1 e montamos os DataFrames `df_corpus` (`docno`, `text`) e `df_test_queries` (`qid`, `query`).

In [15]:
import glob
import os

# Garante que funciona quer o diretório atual seja a raiz do projeto ou Milestone_2
BASE_DIR = (
    "Milestone_2"
    if not os.path.exists("Training") and os.path.exists("Milestone_2")
    else "."
)

# Escolher o modo: "2b" (rápido, 2014) ou "14b" (completo, 2026)
DATASET_MODE = "2b"  # Muda para "14b" para a avaliação final

if DATASET_MODE == "2b":
  TRAIN_PATH = os.path.join(
      BASE_DIR, "Training", "BioASQ_2013_TaskB", "BioASQ-trainingDataset2b.json"
  )
  TEST_DIR = os.path.join(BASE_DIR, "Test", "Task2BGoldenEnriched")
elif DATASET_MODE == "14b":
  TRAIN_PATH = os.path.join(
      BASE_DIR, "Training", "BioASQ-training14b", "training14b.json"
  )
  TEST_DIR = os.path.join(BASE_DIR, "Test", "Task14BGoldenEnriched")
else:
  raise ValueError("DATASET_MODE deve ser '2b' ou '14b'")

# 1. Carregar dados de treino
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
  train_questions = json.load(f)["questions"]

# 2. Carregar todos os batches oficiais de teste (*_golden.json)
test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*_golden.json")))
test_questions = []
for tf in test_files:
  with open(tf, "r", encoding="utf-8") as f:
    test_questions.extend(json.load(f)["questions"])

print(f"Modo ativo: {DATASET_MODE}")
print(f"Ficheiro de treino: {TRAIN_PATH}")
print(f"Ficheiros de teste ({len(test_files)}): {test_files}")
print(
    f"Questões de treino: {len(train_questions)} | Questões de teste:"
    f" {len(test_questions)}"
)

# 3. Construir o corpus agregando snippets de TREINO + TESTE
corpus_dict = {}
for q in train_questions + test_questions:
  for snip in q.get("snippets", []):
    doc_url = snip.get("document", "")
    doc_id = (
        doc_url.split("/")[-1] if "/" in doc_url else doc_url.split("=")[-1]
    )
    text = snip.get("text", "").strip()
    if doc_id and text:
      if doc_id in corpus_dict:
        if text not in corpus_dict[doc_id]:
          corpus_dict[doc_id] += " " + text
      else:
        corpus_dict[doc_id] = text

df_corpus = pd.DataFrame(
    [{"docno": str(doc_id), "text": text} for doc_id, text in corpus_dict.items()]
)
print(f"Total de documentos únicos no corpus: {len(df_corpus)}")

# 4. Construir df_test_queries e qrels_test APENAS com os batches oficiais de teste
test_queries_list = []
qrels_test = {}

for q in test_questions:
  qid = str(q["id"])
  query_text = q["body"].strip()
  gold_docs = [
      d.split("/")[-1] if "/" in d else d.split("=")[-1]
      for d in q.get("documents", [])
  ]
  valid_gold = [str(d) for d in gold_docs if str(d) in corpus_dict]

  if len(valid_gold) > 0:
    qrels_test[qid] = {d: 1 for d in valid_gold}
    test_queries_list.append({"qid": qid, "query": query_text})

df_test_queries = pd.DataFrame(test_queries_list)
print(f"Total de queries de teste avaliáveis: {len(df_test_queries)}")

Modo ativo: 2b
Ficheiro de treino: .\Training\BioASQ_2013_TaskB\BioASQ-trainingDataset2b.json
Ficheiros de teste (5): ['.\\Test\\Task2BGoldenEnriched\\2B1_golden.json', '.\\Test\\Task2BGoldenEnriched\\2B2_golden.json', '.\\Test\\Task2BGoldenEnriched\\2B3_golden.json', '.\\Test\\Task2BGoldenEnriched\\2B4_golden.json', '.\\Test\\Task2BGoldenEnriched\\2B5_golden.json']
Questões de treino: 310 | Questões de teste: 500
Total de documentos únicos no corpus: 7858
Total de queries de teste avaliáveis: 500


### 3. Função de Avaliação de Métricas (Python / NumPy Nnativo)
Cálculo direto de NDCG@5, NDCG@10, MRR@10, MAP e Recall@100 sem depender de compilação C/C++ externa.

In [16]:
def compute_ndcg_at_k(retrieved_docs, gold_docs, k):
    dcg = 0.0
    for i, doc in enumerate(retrieved_docs[:k]):
        if doc in gold_docs:
            dcg += 1.0 / np.log2(i + 2)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(gold_docs), k)))
    return dcg / idcg if idcg > 0 else 0.0

def compute_mrr_at_k(retrieved_docs, gold_docs, k):
    for i, doc in enumerate(retrieved_docs[:k]):
        if doc in gold_docs:
            return 1.0 / (i + 1)
    return 0.0

def compute_map(retrieved_docs, gold_docs):
    hits, sum_prec = 0, 0.0
    for i, doc in enumerate(retrieved_docs):
        if doc in gold_docs:
            hits += 1
            sum_prec += hits / (i + 1)
    return sum_prec / len(gold_docs) if gold_docs else 0.0

def compute_recall_at_k(retrieved_docs, gold_docs, k):
    hits = sum(1 for doc in retrieved_docs[:k] if doc in gold_docs)
    return hits / len(gold_docs) if gold_docs else 0.0

def evaluate_run(run_dict, qrels_dict):
    ndcg5, ndcg10, mrr10, map_scores, recall100 = [], [], [], [], []
    for qid, gold_map in qrels_dict.items():
        if qid not in run_dict:
            continue
        gold_docs = set(gold_map.keys())
        ranked_docs = sorted(run_dict[qid].keys(), key=lambda d: run_dict[qid][d], reverse=True)
        
        ndcg5.append(compute_ndcg_at_k(ranked_docs, gold_docs, 5))
        ndcg10.append(compute_ndcg_at_k(ranked_docs, gold_docs, 10))
        mrr10.append(compute_mrr_at_k(ranked_docs, gold_docs, 10))
        map_scores.append(compute_map(ranked_docs, gold_docs))
        recall100.append(compute_recall_at_k(ranked_docs, gold_docs, 100))
        
    return {
        "NDCG@5": round(float(np.mean(ndcg5)), 4),
        "NDCG@10": round(float(np.mean(ndcg10)), 4),
        "MRR@10": round(float(np.mean(mrr10)), 4),
        "MAP": round(float(np.mean(map_scores)), 4),
        "Recall@100": round(float(np.mean(recall100)), 4)
    }

results_comparison = {}

### 4. Baseline Lexical (BM25 da Milestone 1)
Executamos a busca lexical com os parâmetros otimizados ($k_1=0.6, b=0.75$).

In [17]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))


def preprocess_bm25(text):
  # Minúsculas e remoção de pontuação (mantém apenas letras e números)
  tokens = re.findall(r"\b\w+\b", text.lower())
  # Remoção de stop-words
  return [t for t in tokens if t not in stop_words]


print("A indexar com BM25...")
tokenized_corpus = [preprocess_bm25(doc) for doc in df_corpus["text"]]
bm25 = BM25Okapi(tokenized_corpus, k1=0.6, b=0.75)
doc_ids = df_corpus["docno"].tolist()

print("A avaliar BM25 no conjunto de teste...")
run_bm25 = {}
for _, row in tqdm(
    df_test_queries.iterrows(), total=len(df_test_queries), desc="BM25"
):
  qid = str(row["qid"])
  q_tokens = preprocess_bm25(row["query"])
  scores = bm25.get_scores(q_tokens)

  top_idx = np.argpartition(scores, -100)[-100:]
  top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
  run_bm25[qid] = {doc_ids[idx]: float(scores[idx]) for idx in top_idx}

results_comparison["BM25 (Milestone 1)"] = evaluate_run(run_bm25, qrels_test)
print("Resultados BM25:", results_comparison["BM25 (Milestone 1)"])

A indexar com BM25...
A avaliar BM25 no conjunto de teste...


BM25: 100%|██████████| 500/500 [00:10<00:00, 48.34it/s]

Resultados BM25: {'NDCG@5': 0.6752, 'NDCG@10': 0.6633, 'MRR@10': 0.8167, 'MAP': 0.576, 'Recall@100': 0.8369}


### 5. Recuperação Densa com FAISS (Exemplo do Professor)
Classe `FAISSDenseRetriever` e Bi-Encoder conforme apresentado pelo professor.

In [18]:
class FAISSDenseRetriever:
    """
    Classe para gestão do índice vetorial FAISS e codificação com Bi-Encoder.
    """
    def __init__(self, model_name="BAAI/bge-small-en-v1.5"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f" -> A carregar Bi-Encoder: {model_name} (Dispositivo: {self.device})")
        self.model = SentenceTransformer(model_name, device=self.device)
        self.index = None
        self.docno_list = []

    def build_index(self, df_corpus, batch_size=128):
        print(" -> A gerar embeddings do corpus e a criar índice FAISS...")
        texts = df_corpus["text"].tolist()
        self.docno_list = df_corpus["docno"].tolist()
        
        # Embeddings normalizados para similaridade por cosseno com Inner Product
        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True
        )
        dimension = embeddings.shape[1]
        
        # Construção do Índice FAISS (IndexFlatIP: Produto Interno)
        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(np.array(embeddings, dtype=np.float32))
        print(f" -> Índice FAISS construído com {self.index.ntotal} vetores.")

    def search(self, df_queries, top_k=100):
        print(" -> A pesquisar queries no índice FAISS...")
        queries = df_queries["query"].tolist()
        qids = df_queries["qid"].tolist()
        
        q_embeddings = self.model.encode(
            queries,
            show_progress_bar=True,
            normalize_embeddings=True
        )
        scores, indices = self.index.search(np.array(q_embeddings, dtype=np.float32), top_k)
        
        run_dict = {}
        for i, qid in enumerate(qids):
            run_dict[str(qid)] = {}
            for doc_idx, score in zip(indices[i], scores[i]):
                if doc_idx != -1:
                    run_dict[str(qid)][self.docno_list[doc_idx]] = float(score)
        return run_dict

In [19]:
# 1. Instanciar e construir o índice FAISS com o modelo do professor
faiss_retriever = FAISSDenseRetriever(model_name="BAAI/bge-small-en-v1.5")
faiss_retriever.build_index(df_corpus, batch_size=128)

# 2. Realizar a busca sobre as queries de teste
run_faiss_dense = faiss_retriever.search(df_test_queries, top_k=100)

# 3. Avaliar recuperação densa
results_comparison["Dense FAISS (bge-small-en-v1.5)"] = evaluate_run(run_faiss_dense, qrels_test)
print("Resultados Dense FAISS:", results_comparison["Dense FAISS (bge-small-en-v1.5)"])

 -> A carregar Bi-Encoder: BAAI/bge-small-en-v1.5 (Dispositivo: cpu)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1625.65it/s]


 -> A gerar embeddings do corpus e a criar índice FAISS...


Batches: 100%|██████████| 62/62 [04:31<00:00,  4.38s/it]


 -> Índice FAISS construído com 7858 vetores.
 -> A pesquisar queries no índice FAISS...


Batches: 100%|██████████| 16/16 [00:03<00:00,  4.84it/s]


Resultados Dense FAISS: {'NDCG@5': 0.7556, 'NDCG@10': 0.7434, 'MRR@10': 0.8843, 'MAP': 0.6704, 'Recall@100': 0.9058}


### 6. Comparação Direta: BM25 vs FAISS Dense

In [20]:
df_final_comparison = pd.DataFrame(results_comparison).T
print("=======================================================================")
print("               COMPARAÇÃO: LEXICAL (BM25) vs DENSO (FAISS)             ")
print("=======================================================================")
display(df_final_comparison)

# Guardar resultados em CSV
df_final_comparison.to_csv("dense_vs_bm25_results.csv")
print("Execução concluída com sucesso!")

               COMPARAÇÃO: LEXICAL (BM25) vs DENSO (FAISS)             


,NDCG@5,NDCG@10,MRR@10,MAP,Recall@100
BM25 (Milestone 1),0.6752,0.6633,0.8167,0.5760,0.8369
Dense FAISS (bge-small-en-v1.5),0.7556,0.7434,0.8843,0.6704,0.9058


Execução concluída com sucesso!
